# Telugu Manual Corpus — Archive.org PDFs + EasyOCR (standalone)


In [1]:
!pip install -q internetarchive pymupdf easyocr tqdm

In [2]:
import os
import json
import time
import numpy as np
import pymupdf
import easyocr
import internetarchive as ia
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Define Output Paths in Google Drive
BASE_DIR = '/content/drive/MyDrive/2025201027_LMA_DATASET_TELUGU'
PDF_SAVE_DIR = os.path.join(BASE_DIR, 'archive_pdfs')  # Permanent PDF storage folder
OUTPUT_DIR = os.path.join(BASE_DIR, 'archive_ocr_output')

os.makedirs(PDF_SAVE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# File paths for storage, checkpoints, and stats
OUTPUT_TEXT_FILE = os.path.join(OUTPUT_DIR, 'raw_archive_ocr.txt')
CHECKPOINT_FILE  = os.path.join(OUTPUT_DIR, 'processed_archive_items.json')
STATS_FILE       = os.path.join(OUTPUT_DIR, 'archive_per_book_stats.json')

# 3. Initialize EasyOCR with GPU acceleration
print("Loading EasyOCR Telugu Model on GPU...")
reader = easyocr.Reader(['te', 'en'], gpu=True)
print("EasyOCR loaded successfully!")

def load_checkpoint():
    """Loads the set of successfully processed Archive item identifiers."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            return set(json.load(f))
    return set()

def save_checkpoint(done_set):
    """Saves completed item identifiers to Google Drive."""
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(sorted(list(done_set)), f, ensure_ascii=False, indent=2)

def load_stats():
    """Loads extracted stats per processed book."""
    if os.path.exists(STATS_FILE):
        with open(STATS_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_stats(stats):
    """Saves updated stats to Google Drive."""
    with open(STATS_FILE, 'w', encoding='utf-8') as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)


Mounted at /content/drive
Loading EasyOCR Telugu Model on GPU...
EasyOCR loaded successfully!


In [ ]:
def process_archive_books(query, max_books_to_process=5):
    done = load_checkpoint()
    stats = load_stats()

    print(f"Searching Internet Archive with query: '{query}'...")
    search_results = ia.search_items(query)

    processed_in_this_run = 0

    for item in search_results:
        if processed_in_this_run >= max_books_to_process:
            print(f"Reached batch limit of {max_books_to_process} books for this run.")
            break

        item_id = item['identifier']

        # Skip already completed or attempted items
        if item_id in done:
            continue

        print(f"\n--- Processing [{processed_in_this_run + 1}/{max_books_to_process}]: {item_id} ---")

        # Step 1: Download ONLY the original primary PDF (excluding *_text.pdf)
        try:
            item_obj = ia.get_item(item_id)
            pdf_files_in_item = [
                f['name'] for f in item_obj.files
                if f['name'].lower().endswith('.pdf') and not f['name'].lower().endswith('_text.pdf')
            ]

            target_pattern = pdf_files_in_item[0] if pdf_files_in_item else '*.pdf'

            ia.download(
                item_id,
                glob_pattern=target_pattern,
                destdir=PDF_SAVE_DIR,
                ignore_existing=True,
                no_directory=True,
                verbose=False
            )
        except Exception as e:
            print(f"  Error downloading PDF for {item_id}: {e}")
            done.add(item_id)
            save_checkpoint(done)
            continue

        # Locate the downloaded primary PDF in Drive
        pdf_candidates = [
            os.path.join(PDF_SAVE_DIR, f)
            for f in os.listdir(PDF_SAVE_DIR)
            if item_id in f and f.lower().endswith('.pdf') and not f.lower().endswith('_text.pdf')
        ]

        if not pdf_candidates:
            pdf_candidates = [
                os.path.join(PDF_SAVE_DIR, f)
                for f in os.listdir(PDF_SAVE_DIR)
                if item_id in f and f.lower().endswith('.pdf')
            ]

        if not pdf_candidates:
            print(f"  No downloadable PDF found for {item_id}. Skipping.")
            done.add(item_id)
            save_checkpoint(done)
            continue

        pdf_path = pdf_candidates[0]

        # Step 2: Open PDF and run EasyOCR
        try:
            doc = pymupdf.open(pdf_path)
        except Exception as e:
            print(f"  Could not open PDF file {pdf_path}: {e}")
            done.add(item_id)
            save_checkpoint(done)
            continue

        total_pages = len(doc)
        book_extracted_pages = []
        doc_words = 0
        page_num = -1

        try:
            print(f"  PDF saved to Drive ({total_pages} pages). Starting GPU OCR...")
            for page_num in range(total_pages):
                page = doc[page_num]
                matrix = pymupdf.Matrix(2.0, 2.0)
                pix = page.get_pixmap(matrix=matrix, alpha=False)

                img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)

                result = reader.readtext(img_array, detail=0)
                extracted_text = " ".join(result).strip()

                if extracted_text:
                    book_extracted_pages.append(extracted_text)
                    doc_words += len(extracted_text.split())

                if (page_num + 1) % 10 == 0 or (page_num + 1) == total_pages:
                    print(f"  Page {page_num + 1}/{total_pages} processed.")

            doc.close()

            # Step 3: Write extracted text block to output corpus file
            if book_extracted_pages:
                with open(OUTPUT_TEXT_FILE, 'a', encoding='utf-8') as out_file:
                    out_file.write(f"<<SRC:archive_easyocr>><<FILE:{os.path.basename(pdf_path)}>>\n")
                    out_file.write("\n".join(book_extracted_pages) + "\n<<END>>\n\n")

            done.add(item_id)
            stats[item_id] = {"pages": total_pages, "words": doc_words, "file": os.path.basename(pdf_path)}
            save_checkpoint(done)
            save_stats(stats)

            processed_in_this_run += 1
            print(f"  Finished {item_id}. Words extracted: {doc_words:,}")

        except Exception as e:
            print(f"  Error while processing {item_id} on page {page_num + 1}: {e}")
            print("  Book NOT marked complete; safely skipped for retry.")
            doc.close()

    # Summary Statistics
    total_words_all = sum(s["words"] for s in stats.values())
    print("\n=== Batch OCR Processing Complete ===")
    print(f"Total books completed so far: {len(stats)}")
    print(f"Total words extracted across all books: {total_words_all:,}")
    print(f"Estimated Telugu tokens (~1.7x factor): {int(total_words_all * 1.7):,}")
    print(f"PDFs saved to: {PDF_SAVE_DIR}")
    print(f"Corpus updated at: {OUTPUT_TEXT_FILE}")

# Set query and run batch
SEARCH_QUERY = 'language:Telugu AND mediatype:texts'
process_archive_books(query=SEARCH_QUERY, max_books_to_process=5)

Searching Internet Archive with query: 'language:Telugu AND mediatype:texts'...

--- Processing [1/5]: 0309_2105 ---
  No downloadable PDF found for 0309_2105. Skipping.

--- Processing [1/5]: 04QuranTransl865536535368658ation356 ---
  No downloadable PDF found for 04QuranTransl865536535368658ation356. Skipping.

--- Processing [1/5]: 13_2010_7_8_dd_vv_ss_041_p_prsvm02a102_z_z ---
  PDF saved to Drive (41 pages). Starting GPU OCR...
  Page 10/41 processed.
  Page 20/41 processed.
  Page 30/41 processed.
  Page 40/41 processed.
  Page 41/41 processed.
  Finished 13_2010_7_8_dd_vv_ss_041_p_prsvm02a102_z_z. Words extracted: 7,343

--- Processing [2/5]: 1910-1 ---
  No downloadable PDF found for 1910-1. Skipping.

--- Processing [2/5]: 1945_20251015 ---
  No downloadable PDF found for 1945_20251015. Skipping.

--- Processing [2/5]: 1955_20250909 ---
  No downloadable PDF found for 1955_20250909. Skipping.

--- Processing [2/5]: 1956_20250909 ---
  No downloadable PDF found for 1956_2025090